# Chapter 9 Exercise 3 Solution - Cavity Phase Error

This notebook answers every part of Exercise 3 using actual SciBmad `RFCavity` tracking.
It uses the same low-energy proton and cavity voltage chosen in the Exercise 2 solution.

The exercise asks us to:

1. add a finite cavity phase error;
2. verify that the reference energy remains unchanged while the particle's phase-space
   coordinate `pz` changes;
3. find the range of phase errors that decelerates the particle so strongly that it cannot
   pass through the cavity.

In [1]:
using SciBmad
using Plots
using Printf

## SciBmad interpretation of `phi0_err`

Classic Bmad `lcavity` elements have a separate `phi0_err` attribute. SciBmad currently exposes
`RFCavity`, whose phase is specified by `phi0`, but it does not expose a separate `phi0_err`
attribute. In this solution we represent the same physical phase error by setting

$$
\phi_0=\phi_{0,\mathrm{design}}+\phi_{0,\mathrm{err}}.
$$

This changes the RF phase seen by the tracked particle. It does not change the beamline's
specified reference energy.

## Unified chosen parameters

The exercise does not provide numerical values, so this solution uses the same values as the
Exercise 2 notebook:

$$
z_{\mathrm{in}}=0.01\ \mathrm{m},\qquad
p_{c,\mathrm{in}}=0.5mc^2,\qquad
V=0.5mc^2.
$$

The particle is a proton. The cavity is very short so that the RF phase changes negligibly
during transit. The design phase places the selected particle at the accelerating crest.

In [2]:
species = Species("proton")
mass_energy = massof(species)

# Values shared with the Exercise 2 solution.
z_in = 0.01
pc_in = 0.5 * mass_energy
voltage = 0.5 * mass_energy

# Parameters used to model a very short RF cavity in SciBmad.
cavity_L = 1e-8
rf_frequency = 1e6

E_in = sqrt(pc_in^2 + mass_energy^2)
kinetic_energy_in = E_in - mass_energy
beta_in = pc_in / E_in
omega_rf = 2pi * rf_frequency

# SciBmad uses phi_particle = phi0 - omega*z/(beta*c).
# Therefore this value puts the selected particle at phi_particle = pi/2.
phi0_design = pi/2 + omega_rf * z_in / (beta_in * C_LIGHT)

@printf("Initial total energy   = %.9e eV\n", E_in)
@printf("Initial kinetic energy = %.9e eV = %.6f mc^2\n",
        kinetic_energy_in, kinetic_energy_in / mass_energy)
@printf("Cavity voltage         = %.9e eV = 0.5 mc^2\n", voltage)

Initial total energy   = 1.049020087e+09 eV
Initial kinetic energy = 1.107479972e+08 eV = 0.118034 mc^2
Cavity voltage         = 4.691360447e+08 eV = 0.5 mc^2


The helper below performs a complete SciBmad simulation for one chosen phase error. A successful
`track!` call means the particle reaches the cavity exit. If the particle is decelerated until
its forward momentum vanishes inside the cavity, SciBmad cannot continue the forward tracking;
this appears as a `DomainError`, which the helper records as `passed = false`.

In [3]:
function track_with_phase_error(phi0_err)
    cavity = RFCavity(
        name="PHASE_ERROR_CAVITY",
        L=cavity_L,
        voltage=voltage,
        rf_frequency=rf_frequency,
        phi0=phi0_design + phi0_err,
        tracking_method=SciBmad.SplitIntegration(order=2, num_steps=100),
    )

    line = Beamline([cavity]; species_ref=species, pc_ref=pc_in)
    E_ref_before = line.E_ref
    particle = Bunch(
        [0.0, 0.0, 0.0, 0.0, z_in, 0.0];
        species=line.species_ref,
        p_over_q_ref=line.p_over_q_ref,
    )

    try
        track!(particle, line)
        return (
            passed=true,
            z=particle.coords.v[1, 5],
            pz=particle.coords.v[1, 6],
            E_ref_before=E_ref_before,
            E_ref_after=line.E_ref,
        )
    catch err
        err isa DomainError || rethrow()
        return (
            passed=false,
            z=NaN,
            pz=NaN,
            E_ref_before=E_ref_before,
            E_ref_after=line.E_ref,
        )
    end
end

track_with_phase_error (generic function with 1 method)

## Part 1 - Add a finite phase error

We first compare the design phase with a finite phase error of $30^\circ$.

At the design phase, the selected particle sees the accelerating crest and receives nearly the
full voltage. Adding $30^\circ$ moves the particle away from the crest, so it receives a smaller
energy kick.

In [4]:
design_result = track_with_phase_error(0.0)
finite_error_deg = 30.0
error_result = track_with_phase_error(deg2rad(finite_error_deg))

@printf("Design phase: passed = %s, final pz = %.12f\n",
        design_result.passed, design_result.pz)
@printf("%g degree error: passed = %s, final pz = %.12f\n",
        finite_error_deg, error_result.passed, error_result.pz)
@printf("Change in final pz = %.12f\n", error_result.pz - design_result.pz)

@assert design_result.passed && error_result.passed
@assert error_result.pz != design_result.pz

Design phase: passed = true, final pz = 1.544039299028
30 degree error: passed = true, final pz = 1.371283059100
Change in final pz = -0.172756239928


**Part 1 conclusion:** the finite phase error changes the particle's RF energy kick and therefore
changes its final `pz`. For the chosen values, the $30^\circ$ error reduces the final `pz` because
the particle is no longer exactly at the accelerating crest.

## Part 2 - Verify that reference energy does not change

The reference energy is a beamline property. Changing the cavity phase changes the tracked
particle's energy, but it does not rewrite that beamline property. We verify this by comparing
the reference energy before and after tracking for both phase settings.

In [5]:
@printf("Design phase E_ref before = %.9e eV\n", design_result.E_ref_before)
@printf("Design phase E_ref after  = %.9e eV\n", design_result.E_ref_after)
@printf("30 degree E_ref before    = %.9e eV\n", error_result.E_ref_before)
@printf("30 degree E_ref after     = %.9e eV\n", error_result.E_ref_after)

@assert design_result.E_ref_before == design_result.E_ref_after
@assert error_result.E_ref_before == error_result.E_ref_after
@assert design_result.E_ref_after == error_result.E_ref_after

Design phase E_ref before = 1.049020087e+09 eV
Design phase E_ref after  = 1.049020087e+09 eV
30 degree E_ref before    = 1.049020087e+09 eV
30 degree E_ref after     = 1.049020087e+09 eV


**Part 2 conclusion:** the reference energy is identical before and after tracking and is also
identical for both phase errors. Meanwhile, Part 1 showed that the particle's final `pz` changes.
This is exactly the distinction requested by the exercise.

## Part 3 - Find the phase-error range where the particle cannot pass

Because the design particle is at the accelerating crest, the short-cavity energy kick is
approximately

$$
\Delta E(\phi_{0,\mathrm{err}})=V\cos(\phi_{0,\mathrm{err}}).
$$

The particle cannot continue forward if the cavity removes at least all of its initial kinetic
energy:

$$
E_{\mathrm{in}}+V\cos(\phi_{0,\mathrm{err}})\le mc^2.
$$

Solving the equality gives the analytic boundary of the failure region.

In [6]:
phase_boundary = acos(-kinetic_energy_in / voltage)
phase_boundary_deg = rad2deg(phase_boundary)

@printf("Analytic boundary |phi0_err| = %.6f degrees\n", phase_boundary_deg)
@printf("Analytic failure range: approximately [-180, -%.6f] U [%.6f, 180] degrees\n",
        phase_boundary_deg, phase_boundary_deg)

Analytic boundary |phi0_err| = 103.654585 degrees
Analytic failure range: approximately [-180, -103.654585] U [103.654585, 180] degrees


We now verify the analytic result using actual SciBmad tracking. The scan covers
$-180^\circ\le\phi_{0,\mathrm{err}}\le180^\circ$ in steps of $0.5^\circ$.

For passing particles, the plot shows final `pz`. Missing points are phase errors for which the
particle is decelerated until it cannot complete forward tracking through the cavity. The dashed
lines show the analytic boundaries.

In [7]:
phase_errors_deg = collect(-180.0:0.5:180.0)
scan_results = track_with_phase_error.(deg2rad.(phase_errors_deg))
passed_scan = getproperty.(scan_results, :passed)
pz_scan = getproperty.(scan_results, :pz)

passing_phase_errors = phase_errors_deg[passed_scan]
first_failed_positive = minimum(phase_errors_deg[(.!passed_scan) .& (phase_errors_deg .>= 0)])
last_failed_negative = maximum(phase_errors_deg[(.!passed_scan) .& (phase_errors_deg .<= 0)])

@printf("SciBmad passing range at 0.5 degree resolution: [%.1f, %.1f] degrees\n",
        minimum(passing_phase_errors), maximum(passing_phase_errors))
@printf("SciBmad failure range at 0.5 degree resolution: [-180, %.1f] U [%.1f, 180] degrees\n",
        last_failed_negative, first_failed_positive)

@assert abs(first_failed_positive - phase_boundary_deg) <= 0.5
@assert abs(last_failed_negative + phase_boundary_deg) <= 0.5

SciBmad passing range at 0.5 degree resolution: [-103.5, 103.5] degrees
SciBmad failure range at 0.5 degree resolution: [-180, -104.0] U [104.0, 180] degrees


In [ ]:
phase_plot = plot(
    phase_errors_deg,
    pz_scan;
    xlabel="phi0_err (degrees)",
    ylabel="final pz",
    title="Low-energy proton through the RF cavity",
    label="SciBmad passing particle",
    linewidth=2,
    legend=:bottom,
    left_margin=6Plots.mm,
    bottom_margin=5Plots.mm,
)
vline!(
    phase_plot,
    [-phase_boundary_deg, phase_boundary_deg];
    label="analytic turn-around boundary",
    linestyle=:dash,
    linewidth=2,
)
phase_plot

### Rendered scan result

The following saved figure is the result of the scan above. The curve exists only where the
particle successfully reaches the cavity exit.

![Phase-error scan](../../assets/chapter9_exercise3_phase_error_scan.png)

**Part 3 conclusion:** the analytic boundary is

$$
|\phi_{0,\mathrm{err}}|\simeq103.65^\circ.
$$

With a $0.5^\circ$ scan step, SciBmad finds that the particle passes for approximately
$-103.5^\circ\le\phi_{0,\mathrm{err}}\le103.5^\circ$ and fails to pass for approximately

$$
[-180^\circ,-104^\circ]\ \cup\ [104^\circ,180^\circ].
$$

The numerical tracking result therefore agrees with the direct energy argument.